# MongoDB Query Benchmark

Ten queries against the Yelp aggregate collections, grouped as **simple filtering**,
**complex related-data queries** and **aggregated reports**. Each one is written out
plainly as `db.collection.find(...)` / `db.collection.aggregate(...)` and wrapped in
`bench(...)`, which records its **execution time**.

Per query:
- **`wall_ms_median`** - end-to-end wall-clock ms to run the query and fully
  materialize the results, median of `REPEATS` runs (after `WARMUP` warmups).
  This is the number to compare against an equivalent relational query.
- **`wall_ms_min` / `wall_ms_max`** - fastest and slowest of those same runs. The
  spread between them shows how stable the timing is; a wide gap usually means the
  cache was still filling up.
- **`n_returned`** - how many documents the query produced.

Heavy queries pass a smaller `repeats=` so the whole notebook still finishes in a
few minutes; the value used is visible in each cell and recorded in the results table.

Run the setup cells once, then run each query cell. Re-running a query cell
just updates its row (keyed by name), so timings never duplicate.

In [ ]:
import os, time
from datetime import datetime, timezone
from statistics import median
import pandas as pd
from pymongo import MongoClient

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
MONGO_DB  = os.getenv("MONGO_DB", "yelp")
REPEATS   = 10      # timed runs per query (median is reported)
WARMUP    = 1      # throwaway runs before timing (warms cache)

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
print("Connected:", MONGO_URI, "| db =", MONGO_DB)

## Context - server, collection sizes, query constants

The queries below all target **Philadelphia** and the year **2018**, so those are
pinned here as constants and reused verbatim inside every query.

Note the document shape: the city lives at `location.city` (not at the top level), and
the per-business `review_count` / `stars` are the denormalized values stored on the
business aggregate.

In [ ]:
print("server version:", client.server_info()["version"])
for c in ["businesses", "users", "reviews", "tips"]:
    print(f"  {c:12s}: {db[c].estimated_document_count():,}")

CITY = "Philadelphia"
YEAR_START = datetime(2018, 1, 1, tzinfo=timezone.utc)   # inclusive
YEAR_END   = datetime(2019, 1, 1, tzinfo=timezone.utc)   # exclusive

n_city = db.businesses.count_documents({"location.city": CITY})
print(f"\nCITY  : {CITY}  ({n_city:,} businesses)")
print(f"WINDOW: {YEAR_START:%Y-%m-%d} .. {YEAR_END:%Y-%m-%d} (exclusive)")

## The timer

`bench(name, fn)` runs `fn()` (a lambda returning a cursor) `REPEATS` times, records the
median / min / max of those runs, and **returns the rows of the last run as a
`DataFrame`** - so a query cell that ends in `bench(...)` both records a timing and
displays its own output. Write the actual query in the lambda. Pass `repeats=` /
`warmup=` to make an expensive query cheaper to measure.

In [ ]:
results = []

def bench(name, fn, repeats=REPEATS):
    """Time `fn` and return its result rows as a DataFrame (from the last timed run)."""
    times, rows = [], []
    for _ in range(repeats):
        t0 = time.perf_counter()
        rows = list(fn())
        times.append((time.perf_counter() - t0) * 1000.0)

    results[:] = [r for r in results if r["name"] != name]   # idempotent re-runs
    results.append({"db": "mongodb", "name": name,
                    "wall_ms_median": round(median(times), 2),
                    "wall_ms_min": round(min(times), 2),
                    "wall_ms_max": round(max(times), 2),
                    "n_returned": len(rows), "repeats": repeats})
    print(f"{name:32s}  wall={median(times):9.2f} ms  "
          f"(min {min(times):.2f} / max {max(times):.2f})  n={len(rows)}")
    return pd.DataFrame(rows)

# Simple filtering

Single-collection queries: a filter, a projection and at most a sort.

**1. Businesses in Philadelphia that are currently open and have more than 100 reviews**

`is_open` is stored as `1` / `0`; `review_count` is the denormalized total on the
business document, so no join is needed.

In [ ]:
bench("philly_open_over_100_reviews",
      lambda: db.businesses.find(
          {"location.city": CITY,
           "is_open": 1,
           "review_count": {"$gt": 100}},
          {"_id": 0, "id": 1, "name": 1, "stars": 1, "review_count": 1}
      ).sort([("review_count", -1)]))

**2. Five-star reviews written during 2018**

A range scan over `date` combined with an equality on `stars`. Only the identifying
fields are projected - pulling the full review text for every match would measure
network transfer more than query execution.

In [ ]:
bench("five_star_reviews_2018",
      lambda: db.reviews.find(
          {"stars": 5,
           "date": {"$gte": YEAR_START, "$lt": YEAR_END}},
          {"_id": 0, "id": 1, "business_id": 1, "user_id": 1, "stars": 1, "date": 1}
      )
    )

**3. Businesses in Philadelphia that have a parking lot**

Yelp stores `BusinessParking` as a stringified dict
(`"{'garage': False, 'street': False, 'lot': True, ...}"`), so "has a parking lot"
is a regex on that attribute string.

In [ ]:
bench("philly_with_parking_lot",
      lambda: db.businesses.find(
          {"location.city": CITY,
           "attributes.BusinessParking": {"$regex": r"'lot':\s*True"}},
          {"_id": 0, "id": 1, "name": 1, "attributes.BusinessParking": 1}
      ))

# Complex related-data queries

Queries that have to combine two or three collections with `$lookup`.

**4. Top 10 restaurants in Philadelphia by number of elite-user reviews**

Philadelphia restaurants -> their reviews -> the review authors, keeping only reviews
written by elite users, then counting them and averaging their stars per business.

The reviews are grouped by author *before* the user join, so each distinct reviewer is
looked up once instead of once per review (roughly halves the runtime).

In [ ]:
bench("top_restaurants_by_elite_reviews",
      lambda: db.businesses.aggregate([
          {"$match": {"location.city": CITY, "categories": "Restaurants"}},
          {"$project": {"_id": 0, "id": 1, "name": 1}},
          {"$lookup": {
              "from": "reviews",
              "localField": "id", "foreignField": "business_id",
              "as": "revs",
              "pipeline": [{"$project": {"_id": 0, "user_id": 1, "stars": 1}}]}},
          {"$unwind": "$revs"},
          # one row per reviewer -> one user lookup per distinct reviewer
          {"$group": {"_id": "$revs.user_id",
                      "rated": {"$push": {"business_id": "$id", "name": "$name",
                                          "stars": "$revs.stars"}}}},
          {"$lookup": {
              "from": "users",
              "localField": "_id", "foreignField": "id",
              "as": "elite_user",
              "pipeline": [{"$match": {"elite.is_elite": True}},
                           {"$project": {"_id": 1}}]}},
          {"$match": {"elite_user.0": {"$exists": True}}},
          {"$unwind": "$rated"},
          {"$group": {"_id": {"business_id": "$rated.business_id",
                              "name": "$rated.name"},
                      "elite_reviews": {"$sum": 1},
                      "elite_avg_stars": {"$avg": "$rated.stars"}}},
          {"$project": {"_id": 0,
                        "business_id": "$_id.business_id", "name": "$_id.name",
                        "elite_reviews": 1,
                        "elite_avg_stars": {"$round": ["$elite_avg_stars", 2]}}},
          {"$sort": {"elite_reviews": -1}},
          {"$limit": 10},
      ], allowDiskUse=True)
    )

**5. Users who reviewed at least 3 different businesses in the same city**

Reviews carry no city, so every reviewed business has to be resolved to its city first.
Grouping the reviews per business before that join keeps it to one lookup per business
instead of one per review.

What comes out are `(user, city)` pairs covering 3+ distinct businesses - there are
~100k of them, so the pipeline sorts by breadth and keeps the top 100 **before**
joining the user documents; otherwise the query spends all its time fetching user
aggregates.

In [ ]:
bench("users_3plus_businesses_same_city",
      lambda: db.reviews.aggregate([
          {"$group": {"_id": "$business_id", "users": {"$addToSet": "$user_id"}}},
          {"$lookup": {
              "from": "businesses",
              "localField": "_id", "foreignField": "id",
              "as": "biz",
              "pipeline": [{"$project": {"_id": 0, "name": 1,
                                         "city": "$location.city"}}]}},
          {"$unwind": "$biz"},
          {"$unwind": "$users"},
          {"$group": {"_id": {"user_id": "$users", "city": "$biz.city"},
                      "businesses": {"$addToSet": {"business_id": "$_id",
                                                   "name": "$biz.name"}}}},
          {"$match": {"$expr": {"$gte": [{"$size": "$businesses"}, 3]}}},
          {"$addFields": {"n_businesses": {"$size": "$businesses"}}},
          {"$sort": {"n_businesses": -1}},
          {"$limit": 100},
          {"$lookup": {
              "from": "users",
              "localField": "_id.user_id", "foreignField": "id",
              "as": "user",
              "pipeline": [{"$project": {"_id": 0, "id": 1, "name": 1, "fans": 1,
                                         "review_count": 1, "average_stars": 1}}]}},
          {"$unwind": "$user"},
          {"$project": {"_id": 0, "city": "$_id.city", "user": 1,
                        "n_businesses": 1, "businesses": 1}},
      ], allowDiskUse=True),
    )

**6. Open restaurants in Philadelphia with Monday opening hours and outdoor seating**

Five conditions on a single document: city, category, `is_open`, an `hours.Monday`
entry, and the `OutdoorSeating` attribute (stored as the string `"True"`).

In [ ]:
bench("philly_open_monday_outdoor",
      lambda: db.businesses.find(
          {"location.city": CITY,
           "categories": "Restaurants",
           "is_open": 1,
           "hours.Monday": {"$exists": True, "$ne": None},
           "attributes.OutdoorSeating": "True"},
          {"_id": 0, "id": 1, "name": 1, "stars": 1,
           "hours.Monday": 1, "attributes.OutdoorSeating": 1}
      ).sort([("stars", -1)]))

**7. Complimented tips on Philadelphia restaurants in 2018**

Tips filtered by the date window and `compliment_count >= 1`, then joined to the
business (kept only if it is a Philadelphia restaurant) and to the tip author.

In [ ]:
bench("complimented_tips_philly_2018",
      lambda: db.tips.aggregate([
          {"$match": {"date": {"$gte": YEAR_START, "$lt": YEAR_END},
                      "compliment_count": {"$gte": 1}}},
          {"$lookup": {
              "from": "businesses",
              "localField": "business_id", "foreignField": "id",
              "as": "biz",
              "pipeline": [{"$match": {"location.city": CITY,
                                       "categories": "Restaurants"}},
                           {"$project": {"_id": 0, "name": 1}}]}},
          {"$unwind": "$biz"},          # drops tips whose business did not match
          {"$lookup": {
              "from": "users",
              "localField": "user_id", "foreignField": "id",
              "as": "author",
              "pipeline": [{"$project": {"_id": 0, "name": 1}}]}},
          {"$unwind": "$author"},
          {"$project": {"_id": 0,
                        "author": "$author.name",
                        "business": "$biz.name",
                        "text": 1, "compliment_count": 1, "date": 1}},
          {"$sort": {"compliment_count": -1, "date": 1}},
      ], allowDiskUse=True),
    )

# Aggregated reports

Whole-collection roll-ups: group, average, rank.

**8. Average individual review rating and total reviews per business category**

The average is over individual **reviews**, not over the businesses' stored star
ratings. Reviews are folded per business first (one row per business instead of two
million), then joined to the business categories and re-grouped per category.

In [ ]:
bench("category_avg_review_stars",
      lambda: db.reviews.aggregate([
          {"$group": {"_id": "$business_id",
                      "stars_sum": {"$sum": "$stars"},
                      "n": {"$sum": 1}}},
          {"$lookup": {
              "from": "businesses",
              "localField": "_id", "foreignField": "id",
              "as": "biz",
              "pipeline": [{"$project": {"_id": 0, "categories": 1}}]}},
          {"$unwind": "$biz"},
          {"$unwind": "$biz.categories"},
          {"$group": {"_id": "$biz.categories",
                      "stars_sum": {"$sum": "$stars_sum"},
                      "total_reviews": {"$sum": "$n"}}},
          {"$project": {"_id": 0, "category": "$_id", "total_reviews": 1,
                        "avg_review_stars": {
                            "$round": [{"$divide": ["$stars_sum", "$total_reviews"]}, 3]}}},
          {"$sort": {"total_reviews": -1}},
      ], allowDiskUse=True),
    )

**9. Average number of check-ins per business, for each category**

`checkin_stats.total` is precomputed on the business aggregate, so this is a single
`$unwind` + `$group`. Businesses with no check-ins count as 0 rather than being
skipped, so the average really is per business in the category.

In [ ]:
bench("category_avg_checkins_per_business",
      lambda: db.businesses.aggregate([
          {"$unwind": "$categories"},
          {"$group": {"_id": "$categories",
                      "businesses": {"$sum": 1},
                      "avg_checkins": {"$avg": {"$ifNull": ["$checkin_stats.total", 0]}}}},
          {"$project": {"_id": 0, "category": "$_id", "businesses": 1,
                        "avg_checkins": {"$round": ["$avg_checkins", 2]}}},
          {"$sort": {"avg_checkins": -1}},
      ], allowDiskUse=True))

**10. Top 5 businesses per city by average rating (at least 50 reviews)**

`stars` on the business document *is* its average rating, so the ranking is a sort
followed by a per-city `$push` + `$slice` - MongoDB's equivalent of a windowed
`ROW_NUMBER() <= 5`. Ties are broken by review count.

In [ ]:
bench("top5_per_city_min_50_reviews",
      lambda: db.businesses.aggregate([
          {"$match": {"review_count": {"$gte": 50}}},
          {"$sort": {"stars": -1, "review_count": -1}},
          {"$group": {"_id": "$location.city",
                      "ranked": {"$push": {"business_id": "$id", "name": "$name",
                                           "stars": "$stars",
                                           "review_count": "$review_count"}}}},
          {"$project": {"_id": 0, "city": "$_id",
                        "top_5": {"$slice": ["$ranked", 5]}}},
          {"$sort": {"city": 1}},
      ], allowDiskUse=True))

## Results - table, CSV, chart

In [ ]:
results_df = pd.DataFrame(results)
os.makedirs("data/benchmarks", exist_ok=True)
results_df.to_csv("data/benchmarks/mongo_query_times.csv", index=False)
print("saved -> data/benchmarks/mongo_query_times.csv")

try:
    import matplotlib.pyplot as plt
    d = results_df.dropna(subset=["wall_ms_median"]).sort_values("wall_ms_median")
    # asymmetric whiskers: median back to min, median out to max
    err = [(d["wall_ms_median"] - d["wall_ms_min"]).clip(lower=0),
           (d["wall_ms_max"] - d["wall_ms_median"]).clip(lower=0)]
    plt.figure(figsize=(9, 6))
    plt.barh(d["name"], d["wall_ms_median"], xerr=err, capsize=3,
             error_kw={"elinewidth": 1, "ecolor": "black"})
    plt.xscale("log")
    plt.xlabel("median wall-clock ms (log scale, whiskers = min/max, lower = faster)")
    plt.title(f"MongoDB query execution time (db={MONGO_DB})")
    plt.tight_layout()
    plt.savefig("data/benchmarks/mongo_query_times.png", dpi=120)
    plt.show()
except Exception as e:
    print("plot skipped:", e)

results_df